In [3]:
import torch
import torch.nn as nn
import torch.fx
from torch.fx.experimental.proxy_tensor import make_fx
import math
import pandas as pd

# --- 1. The User's Module (Modified slightly to be standalone) ---
class LizardAttention(nn.Module): # Changed parent from LizardFramework to nn.Module
    def __init__(self, d_model, n_heads, window_size=64, alpha=1.0, m=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.window_size = window_size
        self.alpha = alpha
        self.m = m
        self.meta_tokens = nn.Parameter(torch.randn(1, n_heads, m, self.d_head))
        self.W_gamma = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head))
        self.phi_q = nn.Identity()
        self.phi_k = nn.Identity()

    def forward(self, q, k, v, x=None, alpha=None, triton_kernel=False):
        if alpha is None:
            alpha = self.alpha
        if x is None:
            x = q
        if not triton_kernel:
            gla_out = self.gla_fwd(q, k, v, x)
            awa_out = self.awa_fwd(q, k, v)
            result = gla_out + alpha * awa_out
            return result
        raise NotImplementedError("custom kernel is not implemented yet!")

    def awa_fwd(self, q, k, v):
        batch, heads, seq_len, d_head = q.shape
        meta_tokens = self.meta_tokens.expand(batch, -1, -1, -1)
        out = torch.zeros_like(q)

        for i in range(seq_len):
            start = max(0, i - self.window_size + 1)
            end = min(seq_len, i + self.window_size)
            q_i = q[:, :, i : i + 1, :]
            k_window = k[:, :, start:end, :]
            v_window = v[:, :, start:end, :]
            scores_k = torch.matmul(q_i, k_window.transpose(-2, -1)) / math.sqrt(d_head)
            exp_scores_k = torch.exp(scores_k)
            scores_t = torch.matmul(q_i, meta_tokens.transpose(-2, -1)) / math.sqrt(d_head)
            exp_scores_t = torch.exp(scores_t)
            sum_meta = exp_scores_t.sum(dim=-1, keepdim=True)
            sum_keys = exp_scores_k.sum(dim=-1, keepdim=True)
            denominator = sum_meta + sum_keys
            numerator = torch.matmul(exp_scores_k, v_window)
            out[:, :, i : i + 1, :] = numerator / (denominator + 1e-8)
        return out

    def gla_fwd(self, q, k, v, x):
        batch, heads, seq_len, d_head = q.shape
        q_feat = self.phi_q(q)
        k_feat = self.phi_k(k)
        gamma = torch.sigmoid((self.W_gamma * x).sum(dim=-1, keepdim=True))
        out = torch.zeros_like(q)

        for i in range(seq_len):
            numerator = torch.zeros(batch, heads, 1, d_head, device=q.device, dtype=q.dtype)
            denominator = torch.zeros(batch, heads, 1, 1, device=q.device, dtype=q.dtype)
            q_i = q_feat[:, :, i : i + 1, :]
            for t in range(seq_len):
                if t < i:
                    cum_gamma = torch.prod(gamma[:, :, t + 1 : i + 1, :], dim=2, keepdim=True)
                elif t > i:
                    cum_gamma = torch.prod(gamma[:, :, i + 1 : t + 1, :], dim=2, keepdim=True)
                else:
                    cum_gamma = torch.ones(batch, heads, 1, 1, device=q.device, dtype=q.dtype)

                k_t = k_feat[:, :, t : t + 1, :]
                v_t = v[:, :, t : t + 1, :]
                kv_t = k_t.transpose(-2, -1) @ v_t
                numerator += cum_gamma.squeeze(-1).unsqueeze(-1) * (q_i @ kv_t).squeeze(2).unsqueeze(2)
                qk_t = q_i @ k_t.transpose(-2, -1)
                denominator += cum_gamma * qk_t
            out[:, :, i : i + 1, :] = numerator / (denominator + 1e-8)
        return out

# --- 2. Helper to Generate Table ---
def generate_op_table(graph_module, name="Module"):
    nodes_data = []
    for node in graph_module.graph.nodes:
        # Format arguments for readability
        args_str = str(node.args)
        if len(args_str) > 50: args_str = args_str[:47] + "..."

        nodes_data.append({
            "Node Name": node.name,
            "Op Code": node.op,
            "Target": str(node.target).replace("built-in function ", ""),
            "Args": args_str,
        })

    df = pd.DataFrame(nodes_data)
    print(f"\n--- Operations Table: {name} ---")
    print(df.to_markdown(index=False, tablefmt="grid"))
    return df

# --- 3. Execution & Tracing ---

# Settings (Keep seq_len small to prevent the table from being 1000s of lines)
BATCH, HEADS, SEQ_LEN, D_MODEL = 1, 2, 4, 8
model = LizardAttention(d_model=D_MODEL, n_heads=HEADS, window_size=2)
model.eval()

# Create Dummy Inputs
q = torch.randn(BATCH, HEADS, SEQ_LEN, D_MODEL // HEADS)
k = torch.randn(BATCH, HEADS, SEQ_LEN, D_MODEL // HEADS)
v = torch.randn(BATCH, HEADS, SEQ_LEN, D_MODEL // HEADS)
x = torch.randn(BATCH, HEADS, SEQ_LEN, D_MODEL // HEADS)

# 1. Trace `awa_fwd`
# We use make_fx to trace the execution path with these specific shapes
print("Tracing AWA Forward...")
gm_awa = make_fx(lambda q, k, v: model.awa_fwd(q, k, v))(q, k, v)
generate_op_table(gm_awa, "AWA Forward")

# 2. Trace `gla_fwd`
print("\nTracing GLA Forward...")
gm_gla = make_fx(lambda q, k, v, x: model.gla_fwd(q, k, v, x))(q, k, v, x)
generate_op_table(gm_gla, "GLA Forward")

# 3. Trace `forward` (The whole module)
print("\nTracing Full Module Forward...")
# Note: forward has optional args, so we pass them explicitly for the trace
gm_full = make_fx(lambda q, k, v, x: model(q, k, v, x))(q, k, v, x)
generate_op_table(gm_full, "LizardAttention Full Forward")

Tracing AWA Forward...

--- Operations Table: AWA Forward ---
+------------------+---------------+---------------------------+-------------------------------------+
| Node Name        | Op Code       | Target                    | Args                                |
+==================+===============+===========================+=====================================+
| q_1              | placeholder   | q_1                       | ()                                  |
+------------------+---------------+---------------------------+-------------------------------------+
| k_1              | placeholder   | k_1                       | ()                                  |
+------------------+---------------+---------------------------+-------------------------------------+
| v_1              | placeholder   | v_1                       | ()                                  |
+------------------+---------------+---------------------------+-------------------------------------+
| _param_co

,Node Name,Op Code,Target,Args
0,q_1,placeholder,q_1,()
1,k_1,placeholder,k_1,()
2,v_1,placeholder,v_1,()
3,x_1,placeholder,x_1,()
4,_param_constant0,get_attr,_param_constant0,()
...,...,...,...,...
699,slice_68,call_function,aten.slice.Tensor,"(zeros_like_1, 2, 3, 4)"
700,copy__7,call_function,aten.copy_.default,"(slice_68, div_15)"
701,mul_33,call_function,aten.mul.Tensor,"(zeros_like_1, 1.0)"
702,add_12,call_function,aten.add.Tensor,"(zeros_like, mul_33)"
